# Convert one PDF or JPEG to Excel

Set INPUT_FILENAME to a PDF, JPG, or JPEG in data, add your Datalab API key, and run the cells in order. The notebook saves Datalab's raw JSON response and an Excel workbook in outputs.

In [ ]:
# Uncomment and run each line once if the package is missing.
# %pip install jupyterlab
# %pip install pandas
# %pip install requests
# %pip install lxml
# %pip install openpyxl

In [ ]:
from io import StringIO
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests

INPUT_FILENAME = "example.pdf"  # Put this file in data.
API_KEY = ""  # Paste your Datalab API key here for a personal local copy.
POLL_SECONDS = 5

INPUT_PATH = Path("data") / INPUT_FILENAME
OUTPUT_FOLDER = Path("outputs")
MIME_TYPES = {
    ".pdf": "application/pdf",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
}

API_KEY = API_KEY or os.getenv("DATALAB_API_KEY", "")
if not API_KEY:
    raise ValueError("Paste your Datalab API key into API_KEY or set DATALAB_API_KEY.")
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"File not found: {INPUT_PATH}")
if INPUT_PATH.suffix.lower() not in MIME_TYPES:
    raise ValueError("INPUT_FILENAME must end in .pdf, .jpg, or .jpeg.")


In [ ]:
with INPUT_PATH.open("rb") as input_file:
    response = requests.post(
        "https://www.datalab.to/api/v1/convert",
        headers={"X-API-Key": API_KEY},
        files={"file": (INPUT_PATH.name, input_file, MIME_TYPES[INPUT_PATH.suffix.lower()])},
        data={"output_format": "json", "mode": "accurate"},
        timeout=60,
    )
response.raise_for_status()
check_url = response.json()["request_check_url"]

print(f"Waiting for Datalab to process {INPUT_PATH.name}...")
while True:
    result = requests.get(check_url, headers={"X-API-Key": API_KEY}, timeout=60).json()
    status = result["status"]
    print(f"Status: {status}")

    if status == "complete":
        break
    if status in {"failed", "error"}:
        raise RuntimeError(f"Datalab processing failed: {result}")
    time.sleep(POLL_SECONDS)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
json_path = OUTPUT_FOLDER / f"{INPUT_PATH.stem}_datalab.json"
json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved Datalab JSON: {json_path}")


In [ ]:
def find_tables(item):
    if isinstance(item, dict):
        if item.get("block_type") == "Table":
            yield item
        for value in item.values():
            yield from find_tables(value)
    elif isinstance(item, list):
        for value in item:
            yield from find_tables(value)


def flatten_columns(dataframe):
    if not isinstance(dataframe.columns, pd.MultiIndex):
        return dataframe

    dataframe = dataframe.copy()
    dataframe.columns = [
        " | ".join(str(value) for value in column if not str(value).startswith("Unnamed"))
        for column in dataframe.columns
    ]
    return dataframe


dataframes = [
    pd.read_html(StringIO(table["html"]))[0]
    for table in find_tables(result["json"])
    if table.get("html")
]

if not dataframes:
    raise ValueError("No tables were found in this file.")

excel_path = OUTPUT_FOLDER / f"{INPUT_PATH.stem}_tables.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    for index, dataframe in enumerate(dataframes, start=1):
        flatten_columns(dataframe).to_excel(writer, sheet_name=f"Table_{index}", index=False)

print(f"Saved Excel workbook: {excel_path}")
